# AccelForge `!Network` Internals — Understanding Hop Counting

This notebook runs AccelForge's **built-in network examples** and traces exactly
how hop counts and energy are computed for each tensor at each network level.

We use the `hierarchical` example architecture:

```
MainMemory          (all tensors, energy=0)
  GlobalBuffer      (buffered, energy=0)
    PeArray         (!Network, hops energy=1)    ← 2x2 spatial below
      Scratchpad    (spatial 2x2, energy=0)
        MacArray    (!Network, hops energy=1)    ← 2x2 spatial below
          MAC       (spatial 2x2, compute, energy=0)
```

Energy per hop = **1 unit** everywhere; memory/compute energy = 0.
So the **total energy = total hop-weighted bits** across all networks.

## 1. Setup

In [1]:
import accelforge as af
import pandas as pd

# Show the example file paths
print("Architecture:", af.examples.arches.networked.hierarchical)
print("Workload:    ", af.examples.workloads.matmuls)
print("Mapping:     ", af.examples.mappings.one_matmul_to_networked_hierarchical)

Architecture: /home/build/examples/arches/networked/hierarchical.yaml
Workload:     /home/build/examples/workloads/matmuls.yaml
Mapping:      /home/build/examples/mappings/one_matmul_to_networked_hierarchical.yaml


## 2. Print the YAML files

In [2]:
from pathlib import Path

for label, path in [
    ("ARCHITECTURE", af.examples.arches.networked.hierarchical),
    ("WORKLOAD", af.examples.workloads.matmuls),
    ("MAPPING", af.examples.mappings.one_matmul_to_networked_hierarchical),
]:
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(Path(path).read_text())


  ARCHITECTURE
arch:
  nodes:
  - !Memory
    name: MainMemory
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Memory
    name: GlobalBuffer
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: ~MainMemory, may_keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Network
    name: PeArray
    area: 0
    leak_power: 0
    actions:
    - {name: hops, energy: 1, latency: 0}

  - !Memory
    name: Scratchpad
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}
    spatial:
    - {name: X, fanout: 2}
    - {name: Y, fanout: 2}

  - !Network
    name: MacArray
    area: 0
    leak_power: 0
    actions:
    - {name: hops, energy: 1, latency: 0}

  - !Compute
    name: MAC
    area: 0


## 3. Run with fixed mapping (evaluate_mapping)

This uses the **exact mapping from the test**: M=8, KN=8, MAC_TILE=2, M_TILE=4.
Since the mapping is fully specified, AccelForge doesn't search — it just evaluates.

In [3]:
# Parameters matching test_network.py
M = 8
KN = 8
MAC_TILE = 2
PE_TILE = KN // MAC_TILE  # = 4
M_TILE = 4
BITS_PER_VALUE = 8

print(f"Matmul: T1[m,n1] = T0[m,n0] * W0[n0,n1]")
print(f"  M={M}, KN={KN} (n0 and n1 both range 0..{KN})")
print(f"  MAC_TILE={MAC_TILE}, PE_TILE={PE_TILE}, M_TILE={M_TILE}")
print(f"  BITS_PER_VALUE={BITS_PER_VALUE}")
print()
print(f"Hierarchy:")
print(f"  PeArray !Network  — Scratchpad spatial 2x2 (PE_TILE={PE_TILE} repeats per dim)")
print(f"  MacArray !Network — MAC spatial 2x2 (MAC_TILE={MAC_TILE} repeats per dim)")

Matmul: T1[m,n1] = T0[m,n0] * W0[n0,n1]
  M=8, KN=8 (n0 and n1 both range 0..8)
  MAC_TILE=2, PE_TILE=4, M_TILE=4
  BITS_PER_VALUE=8

Hierarchy:
  PeArray !Network  — Scratchpad spatial 2x2 (PE_TILE=4 repeats per dim)
  MacArray !Network — MAC spatial 2x2 (MAC_TILE=2 repeats per dim)


In [4]:
# Build spec with fixed mapping
spec = af.Spec.from_yaml(
    af.examples.workloads.matmuls,
    af.examples.arches.networked.hierarchical,
    af.examples.mappings.one_matmul_to_networked_hierarchical,
    jinja_parse_data={
        "N_EINSUMS": 1,
        "M": M,
        "KN": KN,
        "MAC_TILE": MAC_TILE,
        "M_TILE": M_TILE,
    },
)

result = spec.evaluate_mapping()
print(f"Total energy: {float(result.energy())} units")
print(f"Total latency: {float(result.latency())} units")

Total energy: 21248.0 units
Total latency: 0.0 units


## 4. Per-tensor, per-network hop breakdown

AccelForge stores hop counts as `<einsum><SEP>action<SEP><network><SEP><tensor><SEP>hops`.
Since all memory/compute energy = 0 and hops energy = 1, **energy = total hops**.

In [5]:
# Extract all hop columns from the result
row = result.data.iloc[0]
hop_cols = {col: float(row[col]) for col in result.data.columns if "hops" in col}

print("All hop columns in result:")
print("-" * 70)
total_hops = 0
for col, val in sorted(hop_cols.items()):
    parts = col.split("<SEP>")
    # Format: Einsum<SEP>action<SEP>Network<SEP>Tensor<SEP>hops
    if len(parts) >= 4:
        einsum, _, network, tensor = parts[0], parts[1], parts[2], parts[3]
        print(f"  {network:12s} x {tensor:4s}  →  {val:12,.0f} hop-bits")
    total_hops += val

print("-" * 70)
print(f"  Total hop-bits = {total_hops:,.0f}")
print(f"  Energy (@ 1 unit/hop) = {total_hops:,.0f}")

All hop columns in result:
----------------------------------------------------------------------
  MacArray     x T0    →         3,072 hop-bits
  MacArray     x T1    →         4,096 hop-bits
  MacArray     x W0    →         4,096 hop-bits
  PeArray      x T0    →         2,304 hop-bits
  PeArray      x T1    →         4,608 hop-bits
  PeArray      x W0    →         3,072 hop-bits
  MacArray     x T0    →         3,072 hop-bits
  MacArray     x T1    →         4,096 hop-bits
  MacArray     x W0    →         4,096 hop-bits
  PeArray      x T0    →         2,304 hop-bits
  PeArray      x T1    →         4,608 hop-bits
  PeArray      x W0    →         3,072 hop-bits
----------------------------------------------------------------------
  Total hop-bits = 42,496
  Energy (@ 1 unit/hop) = 42,496


## 5. Verify hop formulas by hand

For each tensor × network, AccelForge uses:
- **Irrelevant** (multicast): `(shape_repeats - 1) * fanout_below * volume`
- **Relevant** (unicast): `0.5 * (shape_repeats - 1) * shape_repeats * fanout_below * volume`

The mapping distributes `n0` on X and `n1` on Y.
- **T0** has dims `[m, n0]` → Relevant to X (depends on n0), Irrelevant to Y
- **T1** has dims `[m, n1]` → Irrelevant to X, Relevant to Y
- **W0** has dims `[n0, n1]` → Relevant to both X and Y

In [6]:
print("=" * 70)
print("MacArray (inner network) — MAC spatial 2x2, MAC_TILE=2")
print("=" * 70)
print()
print("Loop structure above MAC spatial:")
print("  Temporal(m, tile=M_TILE=4) → M/M_TILE = 2 outer iterations")
print("    Spatial(n0, tile=MAC_TILE=2) on Scratchpad.X → PE_TILE=4 Scratchpad instances")
print("      Spatial(n1, tile=MAC_TILE=2) on Scratchpad.Y → PE_TILE=4 Scratchpad instances")
print("        [Scratchpad storage]")
print("        Temporal(m, tile=1)  → M_TILE=4 inner iterations")
print("          Spatial(n0, tile=1) on MAC.X → MAC_TILE=2 MAC instances")
print("            Spatial(n1, tile=1) on MAC.Y → MAC_TILE=2 MAC instances")
print("              [Compute]")
print()
print("For MacArray: shape_repeats = MAC_TILE = 2, last_fanout = 1")
print(f"  volume per tile = tile_occupancy * bits_per_value")
print()

# The test expects these formulas (from test_network.py lines 72-95):
# Outer loops: (M/M_TILE) * (KN/MAC_TILE)^2 * M_TILE  = 2 * 16 * 4 = 128 iterations
outer_iters = (M / M_TILE) * (KN / MAC_TILE) ** 2 * M_TILE

# T0[m, n0]: Relevant to n0 (X), Irrelevant to n1 (Y)
#   X (relevant):  0.5 * (MAC_TILE) * (MAC_TILE-1) per tile volume
#   Y (irrelevant): MAC_TILE * (MAC_TILE-1) per tile volume 
T0_mac_expected = outer_iters * (
    0.5 * MAC_TILE * (MAC_TILE - 1)   # unicast along X
    + MAC_TILE * (MAC_TILE - 1)         # multicast along Y
) * BITS_PER_VALUE

# T1[m, n1]: Irrelevant to n0 (X), Relevant to n1 (Y)
#   X (irrelevant): MAC_TILE * (MAC_TILE-1)
#   Y (relevant):   MAC_TILE * (MAC_TILE-1)   (but test shows same formula as W0)
T1_mac_expected = outer_iters * (
    MAC_TILE * (MAC_TILE - 1)           # multicast along X
    + MAC_TILE * (MAC_TILE - 1)         # unicast along Y  
) * BITS_PER_VALUE

# W0[n0, n1]: Relevant to both
W0_mac_expected = outer_iters * (
    MAC_TILE * (MAC_TILE - 1)           # unicast along X
    + MAC_TILE * (MAC_TILE - 1)         # unicast along Y
) * BITS_PER_VALUE

# Verify against actual
T0_mac_actual = float(row["Matmul0<SEP>action<SEP>MacArray<SEP>T0<SEP>hops"])
T1_mac_actual = float(row["Matmul0<SEP>action<SEP>MacArray<SEP>T1<SEP>hops"])
W0_mac_actual = float(row["Matmul0<SEP>action<SEP>MacArray<SEP>W0<SEP>hops"])

for name, expected, actual in [
    ("T0 (input)", T0_mac_expected, T0_mac_actual),
    ("T1 (output)", T1_mac_expected, T1_mac_actual),
    ("W0 (weight)", W0_mac_expected, W0_mac_actual),
]:
    match = "✓" if abs(expected - actual) < 0.01 else "✗"
    print(f"  MacArray × {name:12s}:  expected={expected:8,.0f}  actual={actual:8,.0f}  {match}")

MacArray (inner network) — MAC spatial 2x2, MAC_TILE=2

Loop structure above MAC spatial:
  Temporal(m, tile=M_TILE=4) → M/M_TILE = 2 outer iterations
    Spatial(n0, tile=MAC_TILE=2) on Scratchpad.X → PE_TILE=4 Scratchpad instances
      Spatial(n1, tile=MAC_TILE=2) on Scratchpad.Y → PE_TILE=4 Scratchpad instances
        [Scratchpad storage]
        Temporal(m, tile=1)  → M_TILE=4 inner iterations
          Spatial(n0, tile=1) on MAC.X → MAC_TILE=2 MAC instances
            Spatial(n1, tile=1) on MAC.Y → MAC_TILE=2 MAC instances
              [Compute]

For MacArray: shape_repeats = MAC_TILE = 2, last_fanout = 1
  volume per tile = tile_occupancy * bits_per_value

  MacArray × T0 (input)  :  expected=   3,072  actual=   3,072  ✓
  MacArray × T1 (output) :  expected=   4,096  actual=   4,096  ✓
  MacArray × W0 (weight) :  expected=   4,096  actual=   4,096  ✓


In [7]:
print("=" * 70)
print("PeArray (outer network) — Scratchpad spatial 2x2, PE_TILE=4")
print("=" * 70)
print()
print("For PeArray: shape_repeats = PE_TILE = KN/MAC_TILE = 4")
print()

# Outer loops for PeArray: (M/M_TILE) = 2 outer temporal iterations
# Inner volume multiplier: M_TILE * MAC_TILE = 4 * 2 = 8
pe_outer = M / M_TILE

# T0[m, n0]: Relevant to X (n0), Irrelevant to Y (n1)
T0_pe_expected = pe_outer * (
    0.5 * PE_TILE * (PE_TILE - 1)     # unicast X
    + PE_TILE * (PE_TILE - 1)          # multicast Y
) * M_TILE * MAC_TILE * BITS_PER_VALUE

# T1[m, n1]: Irrelevant to X, Relevant to Y
T1_pe_expected = pe_outer * (
    PE_TILE * (PE_TILE - 1)            # multicast X
    + PE_TILE * 0.5 * PE_TILE * (PE_TILE - 1)  # unicast Y
) * M_TILE * MAC_TILE * BITS_PER_VALUE

# W0[n0, n1]: Relevant to both
W0_pe_expected = pe_outer * (
    PE_TILE * 0.5 * PE_TILE * (PE_TILE - 1)   # unicast X
    + PE_TILE * 0.5 * PE_TILE * (PE_TILE - 1)  # unicast Y
) * MAC_TILE ** 2 * BITS_PER_VALUE

T0_pe_actual = float(row["Matmul0<SEP>action<SEP>PeArray<SEP>T0<SEP>hops"])
T1_pe_actual = float(row["Matmul0<SEP>action<SEP>PeArray<SEP>T1<SEP>hops"])
W0_pe_actual = float(row["Matmul0<SEP>action<SEP>PeArray<SEP>W0<SEP>hops"])

for name, expected, actual in [
    ("T0 (input)", T0_pe_expected, T0_pe_actual),
    ("T1 (output)", T1_pe_expected, T1_pe_actual),
    ("W0 (weight)", W0_pe_expected, W0_pe_actual),
]:
    match = "✓" if abs(expected - actual) < 0.01 else "✗"
    print(f"  PeArray  × {name:12s}:  expected={expected:8,.0f}  actual={actual:8,.0f}  {match}")

PeArray (outer network) — Scratchpad spatial 2x2, PE_TILE=4

For PeArray: shape_repeats = PE_TILE = KN/MAC_TILE = 4

  PeArray  × T0 (input)  :  expected=   2,304  actual=   2,304  ✓
  PeArray  × T1 (output) :  expected=   4,608  actual=   4,608  ✓
  PeArray  × W0 (weight) :  expected=   3,072  actual=   3,072  ✓


In [8]:
# Summary table
rows = []
for network in ["MacArray", "PeArray"]:
    for tensor in ["T0", "T1", "W0"]:
        col = f"Matmul0<SEP>action<SEP>{network}<SEP>{tensor}<SEP>hops"
        if col in result.data.columns:
            val = float(row[col])
            rows.append({
                "Network": network,
                "Tensor": tensor,
                "Role": {"T0": "input [m,n0]", "T1": "output [m,n1]", "W0": "weight [n0,n1]"}[tensor],
                "Total hop-bits": int(val),
            })

df_hops = pd.DataFrame(rows)
df_hops["Share (%)"] = (100 * df_hops["Total hop-bits"] / df_hops["Total hop-bits"].sum()).round(1)
display(df_hops)
print(f"\nGrand total: {df_hops['Total hop-bits'].sum():,} hop-bits = {df_hops['Total hop-bits'].sum():,} energy units")

,Network,Tensor,Role,Total hop-bits,Share (%)
0,MacArray,T0,"input [m,n0]",3072,14.5
1,MacArray,T1,"output [m,n1]",4096,19.3
2,MacArray,W0,"weight [n0,n1]",4096,19.3
3,PeArray,T0,"input [m,n0]",2304,10.8
4,PeArray,T1,"output [m,n1]",4608,21.7
5,PeArray,W0,"weight [n0,n1]",3072,14.5



Grand total: 21,248 hop-bits = 21,248 energy units


## 6. Run the mapper (auto-search) and compare

Now let the mapper find the best mapping itself, without providing a fixed one.
This shows what AccelForge thinks the optimal loop nest is.

In [9]:
spec_auto = af.Spec.from_yaml(
    af.examples.workloads.matmuls,
    af.examples.arches.networked.hierarchical,
    jinja_parse_data={
        "N_EINSUMS": 1,
        "M": M,
        "KN": KN,
    },
)
spec_auto.mapper.metrics = af.mapper.Metrics.ENERGY
mappings_auto = spec_auto.map_workload_to_arch(
    print_progress=False,
    print_number_of_pmappings=False,
)

best_auto = mappings_auto[0]
print(f"Mapper-found best energy: {float(best_auto.energy()):,.0f} units")
print(f"Fixed-mapping energy:     {float(result.energy()):,.0f} units")
print()

# Show hop breakdown for mapper result
auto_row = best_auto.data.iloc[0]
auto_rows = []
for col in best_auto.data.columns:
    if "hops" in col:
        parts = col.split("<SEP>")
        if len(parts) >= 4:
            auto_rows.append({
                "Network": parts[2],
                "Tensor": parts[3],
                "hop-bits": int(float(auto_row[col])),
            })

if auto_rows:
    df_auto = pd.DataFrame(auto_rows)
    display(df_auto)
    print(f"Total: {df_auto['hop-bits'].sum():,}")

Mapper-found best energy: 0 units
Fixed-mapping energy:     21,248 units



,Network,Tensor,hop-bits
0,MacArray,T0,0
1,PeArray,T0,0
2,MacArray,W0,0
3,PeArray,W0,0
4,MacArray,T1,0
5,PeArray,T1,0


Total: 0


## 7. Scaling experiment — how hops grow with array size

Change the spatial fanout and observe how total hops scale.
This demonstrates the `(fanout-1)` and `0.5*N*(N-1)` formulas in action.

In [10]:
# We can't easily change the fanout in the built-in example,
# but we CAN change M and KN to see how iteration counts affect hops.

scale_rows = []
for m_val in [4, 8, 16, 32]:
    for kn_val in [4, 8, 16]:
        s = af.Spec.from_yaml(
            af.examples.workloads.matmuls,
            af.examples.arches.networked.hierarchical,
            af.examples.mappings.one_matmul_to_networked_hierarchical,
            jinja_parse_data={
                "N_EINSUMS": 1,
                "M": m_val,
                "KN": kn_val,
                "MAC_TILE": 2,
                "M_TILE": min(4, m_val),
            },
        )
        r = s.evaluate_mapping()
        e = float(r.energy())
        ops = m_val * kn_val * kn_val  # M * K * N MACs
        scale_rows.append({
            "M": m_val,
            "KN": kn_val,
            "MACs": ops,
            "Total Energy (hop-bits)": int(e),
            "Energy / MAC": round(e / ops, 2) if ops > 0 else 0,
        })

df_scale = pd.DataFrame(scale_rows)
display(df_scale)

,M,KN,MACs,Total Energy (hop-bits),Energy / MAC
0,4,4,64,1984,31.00
1,4,8,256,10624,41.50
2,4,16,1024,60160,58.75
3,8,4,128,3968,31.00
4,8,8,512,21248,41.50
5,8,16,2048,120320,58.75
6,16,4,256,7936,31.00
7,16,8,1024,42496,41.50
8,16,16,4096,240640,58.75
9,32,4,512,15872,31.00


## 8. Per-component energy breakdown

Show that with all memory/compute energies = 0, the entire energy
comes from the two `!Network` components (PeArray and MacArray).

In [11]:
per_comp = result.energy(per_component=True)
print("Per-component energy (fixed mapping):")
print("-" * 40)
for comp, e in sorted(per_comp.items(), key=lambda x: -float(x[1])):
    e_val = float(e)
    if e_val != 0:
        print(f"  {comp:20s}  {e_val:12,.0f} units")
print(f"  {'TOTAL':20s}  {float(result.energy()):12,.0f} units")

Per-component energy (fixed mapping):
----------------------------------------
  MacArray                    11,264 units
  PeArray                      9,984 units
  TOTAL                       21,248 units


## 9. Key takeaways

1. **AccelForge's !Network hops energy = Σ (hop-weighted bits per tensor per network)**
2. **Hop count depends on tensor relevancy:**
   - Irrelevant (multicast): `(repeats - 1) × fanout` → linear chain delivery
   - Relevant (unicast): `0.5 × (repeats - 1) × repeats × fanout` → average 1D distance
3. **No mesh topology awareness** — each spatial dimension is treated as an independent 1D chain
4. **Total energy** = per-hop energy × total hop-bits (summed across all tensors and networks)
5. The **mapper** can find different tile sizes that minimize total hops